# Days 2–3: representations, frozen embeddings and freeze

Completed workflow generated from `course_code/`; edit the canonical Python modules and rebuild with `python tools/build_course.py`.

Run setup, shared definitions and stage 03 definitions, then the Day 2 checkpoints below. Each notebook contains its Python dependencies; opening it does not clone the repository. This is controlled simulated DNA classification, not real promoter discovery. Students may follow along or watch the instructor demonstration.

See [execution and access instructions](https://github.com/palexiou/ML_Course_2026/blob/main/notebooks/README.md) and the separate scaffolded practicals and instructor answers. Save a Drive copy to retain edits.

The primary dataset samples a fresh 15-base site from a fixed positive-entry PPM at each planting event (0.95 for positives, 0.05 for negatives), at a uniform valid forward-strand start. Defaults: 1,200 balanced 160-base sequences, seed 2026, 720/240/240 splits. Old fixed-string datasets/caches need regeneration; see the notebook guide. The fixed-position interaction and family-memorization controls retain separate rules.

[Open in Colab](https://colab.research.google.com/github/palexiou/ML_Course_2026/blob/main/notebooks/03_representations.ipynb)

This notebook keeps stages 03–05 in **one runtime and working directory**, preserving `course_output/`. Run stage 03 on Day 2; continue with stages 04 and 05 on Day 3. Alternatively transfer the required provenance-checked outputs explicitly; Git does not contain them.

[Open in Colab](https://colab.research.google.com/github/palexiou/ML_Course_2026/blob/main/notebooks/03_representations.ipynb)

The first encoder load needs internet and about 224 MB of weights plus libraries. The pinned checkpoint uses custom model code and has a CC-BY-NC-SA-4.0 license. The default performs a small live extraction and reuses verified caches if supplied. For an explicitly labeled cached-only contingency, change the stage 04 workflow call to `run_foundation(live_example=False)` **before** freeze.

[Open in Colab](https://colab.research.google.com/github/palexiou/ML_Course_2026/blob/main/notebooks/03_representations.ipynb)

**Stage 05 freezes only.** It does not reveal test results. Run stage 03 on Day 2 and continue stages 04–05 on Day 3; Run all would execute all three stages. No executable reveal cell is included: the instructor must separately authorize `reveal_test(confirmation="REVEAL THE FROZEN TEST RESULTS")` at the planned comparison. Do not delete freeze/reveal markers to permit further tuning.


In [ ]:
# Install missing direct dependencies only; existing packages are not silently replaced.
# Exact locally rehearsed versions below are a baseline, not verified Colab compatibility.
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys
requirements = ['numpy==2.3.2', 'pandas==2.3.1', 'scikit-learn==1.7.2', 'matplotlib==3.10.5', 'torch==2.10.0', 'transformers==4.57.1', 'huggingface-hub==0.36.2', 'tokenizers==0.22.2', 'safetensors==0.8.0', 'einops==0.8.2']
missing = []
for requirement in requirements:
    package, expected = requirement.split('==')
    try:
        installed = version(package)
        print(f'{package}: {installed} (local baseline {expected})')
    except PackageNotFoundError:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    print('If prompted, restart the runtime, then rerun this notebook from the top.')


In [ ]:
"""Shared simulation, evaluation, and artifact contracts for the course."""

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

BASE_ORDER = "ACGT"
PRIMARY_GENERATOR_VERSION = "independent-ppm15-forward-v2"
DATASET_SCHEMA_VERSION = 2
# Rows are positions, columns A/C/G/T. Fixed independently of the dataset seed.
DEFAULT_MOTIF_PPM = (
    (0.05, 0.05, 0.10, 0.80),
    (0.70, 0.10, 0.10, 0.10),
    (0.05, 0.05, 0.15, 0.75),
    (0.65, 0.15, 0.10, 0.10),
    (0.50, 0.40, 0.05, 0.05),
    (0.60, 0.10, 0.20, 0.10),
    (0.10, 0.10, 0.10, 0.70),
    (0.10, 0.10, 0.70, 0.10),
    (0.10, 0.65, 0.15, 0.10),
    (0.15, 0.10, 0.65, 0.10),
    (0.55, 0.15, 0.20, 0.10),
    (0.10, 0.60, 0.10, 0.20),
    (0.10, 0.10, 0.75, 0.05),
    (0.10, 0.10, 0.15, 0.65),
    (0.15, 0.55, 0.20, 0.10),
)

@dataclass(frozen=True)
class SimulationConfig:
    seed: int = 2026
    sample_count: int = 1200
    sequence_length: int = 160
    # Array inputs are copied into immutable tuples during validation.
    motif_ppm: tuple[tuple[float, ...], ...] | np.ndarray = DEFAULT_MOTIF_PPM
    positive_plant_probability: float = 0.95
    negative_plant_probability: float = 0.05
    kmer_length: int = 4

    def __post_init__(self) -> None:
        if not 0 <= self.seed < 2**32:
            raise ValueError("seed must be between 0 and 2**32 - 1")
        if self.sample_count < 20 or self.sample_count % 2:
            raise ValueError("sample_count must be an even integer of at least 20")
        matrix = np.array(self.motif_ppm, dtype=float, copy=True)
        if matrix.shape != (15, 4):
            raise ValueError("motif_ppm must have shape (15 positions, 4 ACGT bases)")
        if not np.isfinite(matrix).all() or np.any(matrix <= 0) or np.any(matrix >= 1):
            raise ValueError("motif_ppm entries must be finite, strictly positive and less than one")
        if not np.allclose(matrix.sum(axis=1), 1.0, rtol=0, atol=1e-12):
            raise ValueError("motif_ppm rows must sum to one")
        if np.any(matrix.max(axis=1) < 0.5):
            raise ValueError("Each motif_ppm position needs a leading probability of at least 0.5")
        object.__setattr__(self, "motif_ppm", tuple(tuple(float(p) for p in row) for row in matrix))
        if self.sequence_length < len(self.motif_ppm):
            raise ValueError("sequence_length must be at least the motif length")
        if not 1 <= self.kmer_length <= self.sequence_length:
            raise ValueError("kmer_length must be between 1 and sequence_length")
        for probability in (
            self.positive_plant_probability,
            self.negative_plant_probability,
        ):
            if not 0 <= probability <= 1:
                raise ValueError("motif planting probabilities must be between 0 and 1")


def simulation_settings(config: SimulationConfig) -> dict[str, Any]:
    return {"dataset_variant": "simple_signal", "algorithm_version": PRIMARY_GENERATOR_VERSION,
            "base_order": BASE_ORDER, "background_probabilities": [0.25] * 4,
            "orientation": "forward_only", **asdict(config)}


def settings_fingerprint(settings: dict[str, Any]) -> str:
    return hashlib.sha256(json.dumps(settings, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()


def generation_contract() -> dict[str, Any]:
    return {"schema_version": DATASET_SCHEMA_VERSION,
            "primary_defaults": simulation_settings(SimulationConfig())}


def expected_motif_kmers(config: SimulationConfig, words: Any) -> pd.DataFrame:
    """Average internal-window probability; enrichment denominator is uniform 4^-k.

    Excludes windows crossing site boundaries, label mixtures and full-sequence counts.
    For k > 15 there are no wholly internal windows, so the annotation is undefined.
    """
    k = config.kmer_length
    if k > 15:
        return pd.DataFrame(columns=["kmer", "motif_window_probability", "motif_vs_background_enrichment"])
    matrix = np.asarray(config.motif_ppm)
    records = []
    for word in words:
        if len(word) != k or set(word) - set(BASE_ORDER):
            raise ValueError("Expected ACGT words of the configured k-mer length")
        bases = [BASE_ORDER.index(base) for base in word]
        probability = sum(float(np.prod(matrix[start + np.arange(k), bases]))
                          for start in range(16 - k)) / (16 - k)
        records.append({"kmer": word,
                        "motif_window_probability": probability,
                        "motif_vs_background_enrichment": probability / (0.25 ** k)})
    return pd.DataFrame(records)


def sample_motif_site(random: np.random.Generator, config: SimulationConfig) -> str:
    # Independent draws for every position of every inserted site.
    return "".join(random.choice(list(BASE_ORDER), p=row) for row in config.motif_ppm)


def generate_course_dataset(
    config: SimulationConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    random = np.random.default_rng(config.seed)
    labels = np.repeat([0, 1], config.sample_count // 2)
    random.shuffle(labels)
    sequences = random.choice(list("ACGT"), size=(config.sample_count, config.sequence_length))
    planting_probabilities = np.where(
        labels == 1,
        config.positive_plant_probability,
        config.negative_plant_probability,
    )
    planted = random.random(config.sample_count) < planting_probabilities
    starts = np.full(config.sample_count, -1, dtype=int)
    sites: list[str | None] = [None] * config.sample_count
    for index in np.flatnonzero(planted):
        start = int(random.integers(0, config.sequence_length - len(config.motif_ppm) + 1))
        site = sample_motif_site(random, config)
        sequences[index, start : start + len(config.motif_ppm)] = list(site)
        starts[index] = start
        sites[index] = site
    sample_ids = [f"simple-{config.seed}-{index:05d}" for index in range(config.sample_count)]
    samples = pd.DataFrame(
        {
            "sample_id": sample_ids,
            "group_id": sample_ids,
            "sequence": ["".join(sequence) for sequence in sequences],
            "label": labels,
        }
    )
    if samples["sequence"].duplicated().any():
        raise ValueError("The independent-sample variant contains duplicate sequences")
    truth = pd.DataFrame(
        {"sample_id": sample_ids, "motif_planted": planted,
         "motif_start": pd.array([int(start) if flag else None for start, flag in zip(starts, planted)], dtype="Int64"),
         "sampled_site": sites}
    )
    return samples, truth


def assign_splits(samples: pd.DataFrame, seed: int) -> pd.DataFrame:
    train_indices, heldout_indices = train_test_split(
        samples.index,
        test_size=0.4,
        stratify=samples["label"],
        random_state=seed,
    )
    validation_indices, test_indices = train_test_split(
        heldout_indices,
        test_size=0.5,
        stratify=samples.loc[heldout_indices, "label"],
        random_state=seed,
    )
    result = samples.copy()
    result["split"] = "train"
    result.loc[validation_indices, "split"] = "validation"
    result.loc[test_indices, "split"] = "test"
    if result.groupby("group_id")["split"].nunique().max() != 1:
        raise ValueError("Related sequences cross split boundaries")
    if len(train_indices) + len(validation_indices) + len(test_indices) != len(result):
        raise ValueError("Splits do not cover the course dataset")
    return result



def classification_metrics(labels: Any, probabilities: Any, evaluation_split: str) -> dict[str, Any]:
    if evaluation_split not in {"validation", "test"}:
        raise ValueError("Evaluation split must be validation or test")
    if not np.isfinite(probabilities).all() or np.any(np.asarray(probabilities) < 0) or np.any(np.asarray(probabilities) > 1):
        raise ValueError("Probabilities must be finite and between zero and one")
    predictions = (np.asarray(probabilities) >= 0.5).astype(int)
    return {
        "evaluation_split": evaluation_split, "primary_metric": "roc_auc", "decision_threshold": 0.5,
        "sample_count": len(labels), "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
    }


def validation_metrics(labels: Any, probabilities: Any) -> dict[str, Any]:
    return classification_metrics(labels, probabilities, "validation")


def write_json(path: str | Path, value: Any) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def file_hash(path: str | Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def sample_hash(samples: pd.DataFrame) -> str:
    return hashlib.sha256(samples.to_csv(index=False, lineterminator="\n").encode()).hexdigest()


def save_dataset(root: str | Path, samples: pd.DataFrame, truth: pd.DataFrame, settings: dict[str, Any]) -> None:
    if "motif" in settings or ("motif_ppm" in settings and settings.get("algorithm_version") != PRIMARY_GENERATOR_VERSION):
        raise ValueError("Legacy primary settings: regenerate using simulation_settings(config)")
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    for split, filename in (("train", "train.csv"), ("validation", "validation.csv"), ("test", "protected_test.csv")):
        samples.loc[samples["split"] == split].to_csv(root / filename, index=False)
    samples[["sample_id", "group_id", "split"]].to_csv(root / "split_manifest.csv", index=False)
    truth.to_csv(root / "simulation_truth.csv", index=False)
    write_json(root / "generation.json", {
        "schema_version": DATASET_SCHEMA_VERSION, "settings": settings,
        "settings_sha256": settings_fingerprint(settings), "dataset_sha256": sample_hash(samples),
        "files": {name: file_hash(root / name) for name in
                  ("train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv")},
    })


def validate_generation(root: str | Path) -> dict[str, Any]:
    """Reject legacy generation and tampered data before development/freeze reads."""
    root = Path(root)
    metadata = json.loads((root / "generation.json").read_text(encoding="utf-8"))
    if metadata.get("schema_version") != DATASET_SCHEMA_VERSION:
        raise ValueError("Incompatible dataset metadata; regenerate development artifacts in a new directory")
    settings = metadata["settings"]
    if "motif" in settings:
        raise ValueError("Legacy fixed-motif dataset; regenerate with the PPM generator")
    if settings.get("dataset_variant") == "simple_signal" or "motif_ppm" in settings:
        fields = SimulationConfig.__dataclass_fields__
        config = SimulationConfig(**{key: settings[key] for key in fields})
        if settings_fingerprint(settings) != settings_fingerprint(simulation_settings(config)):
            raise ValueError("Incompatible primary generator algorithm or settings")
    if metadata.get("settings_sha256") != settings_fingerprint(settings):
        raise ValueError("Generation settings fingerprint mismatch")
    names = {"train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv"}
    if set(metadata.get("files", {})) != names:
        raise ValueError("Incomplete dataset file identity metadata")
    for name in names:
        if file_hash(root / name) != metadata["files"][name]:
            raise ValueError(f"Dataset file identity mismatch: {name}")
    return metadata


def read_dataset(root: str | Path, include_test_labels: bool = False) -> pd.DataFrame:
    root = Path(root)
    validate_generation(root)
    partitions = []
    for split, filename in (("train", "train.csv"), ("validation", "validation.csv"), ("test", "protected_test.csv")):
        if split == "test" and not include_test_labels:
            partition = pd.read_csv(root / filename, usecols=lambda column: column != "label")
            partition["label"] = np.nan
        else:
            partition = pd.read_csv(root / filename)
        required = {"sample_id", "group_id", "sequence", "label", "split"}
        non_null = required - ({"label"} if split == "test" and not include_test_labels else set())
        if not required <= set(partition) or partition[list(non_null)].isna().any().any():
            raise ValueError("Dataset columns are missing or contain null values")
        labels_expected = split != "test" or include_test_labels
        if partition.empty or not partition["split"].eq(split).all() or (labels_expected and set(partition["label"]) != {0, 1}):
            raise ValueError("A partition has an incorrect split or lacks both binary classes")
        partitions.append(partition)
    samples = pd.concat(partitions, ignore_index=True)
    if not samples["sample_id"].is_unique or not samples["sequence"].map(lambda sequence: bool(sequence) and set(sequence) <= set("ACGT")).all():
        raise ValueError("Dataset IDs or DNA sequences are invalid")
    if samples.groupby("group_id")["split"].nunique().max() != 1:
        raise ValueError("Related sequences cross split boundaries")
    if samples.groupby("sequence")["split"].nunique().max() != 1:
        raise ValueError("Duplicate DNA sequences cross split boundaries")
    manifest = pd.read_csv(root / "split_manifest.csv")
    expected = samples[["sample_id", "group_id", "split"]].sort_values("sample_id").reset_index(drop=True)
    actual = manifest.sort_values("sample_id").reset_index(drop=True)
    if not expected.equals(actual):
        raise ValueError("Split manifest does not match sample identities")
    samples.attrs["dataset_generation_sha256"] = file_hash(root / "generation.json")
    return samples


def build_interaction_dataset(seed: int = 2026, pair_count: int = 240) -> tuple[pd.DataFrame, pd.DataFrame]:
    if pair_count < 20:
        raise ValueError("At least 20 independent pairs are required")
    random = np.random.default_rng(seed)
    motif_first, motif_second = "TATAAATG", "CGGACGTC"
    flank_left, flank_right = "GGGGGG", "CCCCCC"
    positions = (32, 80)
    records, truth = [], []
    for pair_index in range(pair_count):
        background = random.choice(list("ACGT"), size=160)
        group = f"order-{seed}-{pair_index:05d}"
        for label, motifs in ((1, (motif_first, motif_second)), (0, (motif_second, motif_first))):
            sequence = background.copy()
            for position, motif in zip(positions, motifs):
                block = flank_left + motif + flank_right
                sequence[position:position + len(block)] = list(block)
            sample_id = f"{group}-{label}"
            records.append({"sample_id": sample_id, "group_id": group, "sequence": "".join(sequence), "label": label})
            truth.append({"sample_id": sample_id, "first_block_start": positions[0], "second_block_start": positions[1], "first_motif": motifs[0], "second_motif": motifs[1]})
    samples = pd.DataFrame(records)
    groups = samples["group_id"].unique()
    train_groups, heldout_groups = train_test_split(groups, test_size=0.4, random_state=seed)
    validation_groups, test_groups = train_test_split(heldout_groups, test_size=0.5, random_state=seed)
    samples["split"] = "train"
    samples.loc[samples["group_id"].isin(validation_groups), "split"] = "validation"
    samples.loc[samples["group_id"].isin(test_groups), "split"] = "test"
    return samples, pd.DataFrame(truth)


In [ ]:
"""Inference contracts shared by development and the frozen comparison."""

from pathlib import Path
from typing import Any
import hashlib

import numpy as np
import pandas as pd
import torch
from torch import nn

def one_hot(sequences: Any) -> torch.Tensor:
    encoded = np.stack([np.array(list(sequence))[:, None] == np.array(list("ACGT"))[None, :] for sequence in sequences]).astype(np.float32)
    return torch.from_numpy(encoded.transpose(0, 2, 1))


class SequenceCNN(nn.Module):
    def __init__(self, sequence_length: int = 160) -> None:
        super().__init__()
        self.filters = nn.Sequential(nn.Conv1d(4, 16, kernel_size=7, padding=3), nn.ReLU(), nn.Conv1d(16, 16, kernel_size=5, padding=2), nn.ReLU())
        self.classifier = nn.Sequential(nn.Linear(16 * sequence_length + 16, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, sequences: torch.Tensor) -> torch.Tensor:
        features = self.filters(sequences)
        representation = torch.cat([features.flatten(1), features.amax(dim=2)], dim=1)
        return self.classifier(representation).squeeze(1)


def predict_cnn(model: SequenceCNN, sequences: Any) -> np.ndarray:
    model.eval()
    with torch.inference_mode():
        return torch.sigmoid(model(one_hot(sequences))).numpy()


MODEL_ID = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
REVISION = "81b29e5786726d891dbf929404ef20adca5b36f1"
POOLING = "last_hidden_state_mean_excluding_special_tokens_and_padding"


def sequence_hashes(samples: pd.DataFrame) -> np.ndarray:
    return np.array([hashlib.sha256(sequence.encode("ascii")).hexdigest() for sequence in samples["sequence"]])


def load_embedding_cache(path: str | Path, samples: pd.DataFrame) -> np.ndarray:
    with np.load(path, allow_pickle=False) as cache:
        identity = samples.attrs.get("dataset_generation_sha256")
        if identity is not None and ("dataset_generation_sha256" not in cache or str(cache["dataset_generation_sha256"]) != identity):
            raise ValueError("Embedding cache generation identity mismatch; regenerate the cache")
        ids = cache["sample_ids"].astype(str)
        if len(set(ids)) != len(ids) or set(ids) != set(samples["sample_id"]):
            raise ValueError("Embedding cache sample identities do not match the dataset")
        if str(cache["revision"]) != REVISION or str(cache["model_id"]) != MODEL_ID or str(cache["pooling"]) != POOLING:
            raise ValueError("Embedding cache provenance does not match the declared representation")
        positions = pd.Index(ids).get_indexer(samples["sample_id"])
        if not np.array_equal(cache["sequence_sha256"][positions], sequence_hashes(samples)):
            raise ValueError("Embedding cache sequence hashes do not match the dataset")
        embeddings = cache["embeddings"][positions]
    if embeddings.shape != (len(samples), 512) or not np.isfinite(embeddings).all():
        raise ValueError("Embedding cache shape or values are invalid")
    return embeddings


## Stage 03: Days 2–3: representations, frozen embeddings and freeze

Run the definition cell, then the workflow cell when the instructor reaches this stage. Definitions do not train models. Generated outputs are temporary runtime files.


In [ ]:
"""Compare bag-of-k-mers and a small position-aware CNN without scoring test data."""

from pathlib import Path
from time import perf_counter
from typing import Any
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from torch import nn





def fit_cnn(train: pd.DataFrame, validation: pd.DataFrame, seed: int, epochs: int = 24) -> tuple[SequenceCNN, list[dict[str, Any]]]:
    torch.manual_seed(seed)
    torch.set_num_threads(2)
    model = SequenceCNN(len(train.iloc[0]["sequence"]))
    inputs = one_hot(train["sequence"])
    labels = torch.tensor(train["label"].to_numpy(), dtype=torch.float32)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
    random = torch.Generator().manual_seed(seed)
    history = []
    best_score = -1.0
    best_state = None
    for epoch in range(epochs):
        model.train()
        permutation = torch.randperm(len(train), generator=random)
        total_loss = 0.0
        for indices in permutation.split(64):
            optimizer.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(model(inputs[indices]), labels[indices])
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach()) * len(indices)
        score = validation_metrics(validation["label"], predict_cnn(model, validation["sequence"]))["roc_auc"]
        history.append({"epoch": epoch + 1, "training_loss": total_loss / len(train), "validation_roc_auc": score})
        if score > best_score:
            best_score = score
            best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
    if best_state is None:
        raise RuntimeError("CNN training did not produce a selected validation checkpoint")
    model.load_state_dict(best_state)
    return model, history


def run_representations(output_dir: str | Path = "course_output", seed: int = 2026, epochs: int = 24, show_plot: bool = True) -> pd.DataFrame:
    root = Path(output_dir)
    if any((root / marker).exists() for marker in ("frozen_manifest.json", "test_results.json")):
        raise RuntimeError("Test results have been revealed. Do not tune this course experiment afterward; use a new development directory.")
    config = SimulationConfig(seed=seed)
    samples, truth = generate_course_dataset(config)
    save_dataset(root / "datasets" / "simple", assign_splits(samples, seed), truth, simulation_settings(config))
    interaction, interaction_truth = build_interaction_dataset(seed)
    save_dataset(root / "datasets" / "interaction", interaction, interaction_truth, {"seed": seed, "pair_count": 240, "positions": [32, 80], "matching_flanks": 6, "lesson": "fixed-position motif order, not universal CNN superiority"})
    results, all_predictions, run_records = [], [], {}
    for variant in ("simple", "interaction"):
        started = perf_counter()
        dataset = read_dataset(root / "datasets" / variant)
        train = dataset.loc[dataset["split"] == "train"]
        validation = dataset.loc[dataset["split"] == "validation"]
        destination = root / "models" / variant
        destination.mkdir(parents=True, exist_ok=True)
        if variant == "interaction":
            vectorizer = CountVectorizer(analyzer="char", ngram_range=(4, 4), lowercase=False)
            features = vectorizer.fit_transform(train["sequence"])
            first_rows = train["label"].to_numpy() == 1
            if (features[first_rows] != features[~first_rows]).nnz:
                raise RuntimeError("Paired k-mer bags differ; the intended representation control is invalid")
        models = {
            "logistic_regression": LogisticRegression(max_iter=2000),
            "random_forest": RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=2),
        }
        probabilities_by_method = {}
        for name, classifier in models.items():
            pipeline = Pipeline([("kmers", CountVectorizer(analyzer="char", ngram_range=(4, 4), lowercase=False)), ("classifier", classifier)])
            pipeline.fit(train["sequence"], train["label"])
            probabilities = pipeline.predict_proba(validation["sequence"])[:, 1]
            joblib.dump(pipeline, destination / f"{name}.joblib")
            probabilities_by_method[name] = probabilities
        cnn, history = fit_cnn(train, validation, seed, epochs)
        probabilities_by_method["cnn"] = predict_cnn(cnn, validation["sequence"])
        torch.save({"state_dict": cnn.state_dict(), "sequence_length": 160}, destination / "cnn.pt")
        pd.DataFrame(history).to_csv(destination / "cnn_history.csv", index=False)
        for name, probabilities in probabilities_by_method.items():
            metrics = validation_metrics(validation["label"], probabilities)
            results.append({"variant": variant, "method": name, **metrics})
            prediction_frame = validation[["sample_id", "group_id", "label"]].copy()
            prediction_frame["probability"] = probabilities
            prediction_frame["variant"] = variant
            prediction_frame["method"] = name
            all_predictions.append(prediction_frame)
        if variant == "interaction":
            controlled = validation[["sample_id", "group_id", "label"]].copy()
            for name, probabilities in probabilities_by_method.items():
                controlled[f"{name}_probability"] = probabilities
            controlled.to_csv(destination / "controlled_order_changes.csv", index=False)
        run_records[variant] = {"dataset_sha256": sample_hash(dataset), "dataset_generation_sha256": file_hash(root / "datasets" / variant / "generation.json"), "training_ids": train["sample_id"].tolist(), "validation_ids": validation["sample_id"].tolist(), "seconds": perf_counter() - started, "cnn_epochs": epochs, "cnn_selection": "highest validation ROC-AUC, earliest tie", "test_evaluated": False}
    result = pd.DataFrame(results)
    result.to_csv(root / "representation_validation.csv", index=False)
    pd.concat(all_predictions, ignore_index=True).to_csv(root / "representation_validation_predictions.csv", index=False)
    write_json(root / "representation_run.json", {"seed": seed, "torch_version": torch.__version__, "device": "cpu", "runs": run_records, "interpretation": "The engineered interaction pairs have identical 4-mer counts. This CNN preserves position through flattening; global pooling alone would discard that cue. Fixed positions and matching flanks are a disclosed teaching construction, not evidence of real promoter prediction."})
    figure, axis = plt.subplots(figsize=(8, 4))
    result.pivot(index="variant", columns="method", values="roc_auc").plot.bar(ax=axis, ylim=(0, 1.05), ylabel="Validation ROC-AUC", rot=0)
    axis.axhline(0.5, linestyle="--", color="gray")
    figure.tight_layout()
    figure.savefig(root / "representation_validation.png", dpi=150)
    if show_plot:
        plt.show()
    plt.close(figure)
    print(result[["variant", "method", "roc_auc", "average_precision"]].to_string(index=False))
    print("Validation only. Test data remains unscored. Paired order control is intentionally engineered.")
    return result


## Day 2 · Section 5: train, inspect and compare

Route: **GitHub → Open in Colab → setup → shared definitions → stage 03 definitions
→ this checkpoint**. Instructor demonstration and optional follow-along are both valid.
Setup and computation are inside the lesson; timings require classroom rehearsal.
Do not run all: stages 04–05 belong to Day 3. No homework or submission.

Run the supplied workflow **once**, then inspect without refitting. It fits three
methods on both dataset variants. Initially we display only the primary PPM experiment.
The declared CPU run is seed 2026, 24 epochs, batch 64, Adam 0.002, two threads.
One-hot input is N×4×160 in A/C/G/T order; one logit per sequence, sigmoid at inference.


In [ ]:
d2_source_identity = {'course_core.py': 'ece9dac6edebf6296e0ba3fae6d6c665ceb8a26b35f517c11591924b7f40c7b0', 'course_inference.py': '5f08b87c810bded9fd170f9a2f2c9a717e981f8cfe827c6241e21630d902ea35', '03_representations.py': '037536591c2dfd918372d82afbd60b083cf82554ba9bff6b04c9ef1e527f9740', '04_foundation_embeddings.py': 'f9db3c1ce415a8d3303f6e1d63e4c22ff497783ea91411d69da0daeaafea0378', '05_frozen_comparison.py': '9863750110e06d5084e01cf4bb14a2a103c89abe16b05df4de0f238dbb7a9978'}
d2_root = Path("course_output")


In [ ]:
# Re-running this cell reads a completed compatible run; it does not fit again.
from contextlib import redirect_stdout
import io

if any((d2_root / name).exists() for name in ("frozen_manifest.json", "test_results.json")):
    raise RuntimeError("This workspace is frozen/revealed; do not restart development here.")
d2_identity_path = d2_root / "day2_teaching_identity.json"
d2_identity = {"sources": d2_source_identity, "seed": 2026, "epochs": 24}
if (d2_root / "representation_run.json").exists():
    if not d2_identity_path.exists() or json.loads(d2_identity_path.read_text()) != d2_identity:
        raise ValueError("Existing run has a different or missing teaching identity. Preserve it; restore a complete compatible run or use a fresh runtime.")
    print("Reading the existing completed run; no fitting repeated.")
else:
    if d2_root.exists() and any(d2_root.iterdir()):
        raise ValueError("Incomplete/existing workspace: preserve it and use a fresh runtime.")
    print("Running stage 03 once (both variants); inspect primary results first.")
    with redirect_stdout(io.StringIO()):
        run_representations(output_dir=d2_root, seed=2026, epochs=24, show_plot=False)
    write_json(d2_identity_path, d2_identity)
    print("Stage 03 complete. Validation only; continue below.")


### Read the primary run

Check dataset/source identity and row alignment before interpreting the result.
Training loss averages minibatch losses during updates. Validation ROC-AUC selects
the earliest highest-scoring epoch after **all 24 epochs**, not early stopping.
Test labels/results do not guide any decision. File hashes verify identity only.


In [ ]:
from collections import Counter
import joblib

d2_run = json.loads((d2_root / "representation_run.json").read_text())
assert d2_run["seed"] == 2026 and d2_run["device"] == "cpu"
assert json.loads(d2_identity_path.read_text()) == d2_identity
d2_metrics = pd.read_csv(d2_root / "representation_validation.csv")
d2_predictions = pd.read_csv(d2_root / "representation_validation_predictions.csv")
d2_data = {}
d2_models = {}
d2_histories = {}
d2_selected = {}
d2_methods = ["logistic_regression", "random_forest", "cnn"]
assert set(d2_metrics.variant) == {"simple", "interaction"}
assert set(d2_predictions.variant) == {"simple", "interaction"}
for d2_variant, d2_size in [("simple", 240), ("interaction", 96)]:
    d2_dataset_dir = d2_root / "datasets" / d2_variant
    d2_generation = validate_generation(d2_dataset_dir)
    d2_record = d2_run["runs"][d2_variant]
    assert d2_record["dataset_generation_sha256"] == file_hash(d2_dataset_dir / "generation.json")
    # Generation digest binds complete saved files; run dataset hash has masked test labels.
    assert not d2_record["test_evaluated"] and d2_record["cnn_epochs"] == 24
    assert d2_record["cnn_selection"] == "highest validation ROC-AUC, earliest tie"
    if d2_variant == "simple":
        assert d2_generation["settings_sha256"] == settings_fingerprint(simulation_settings(SimulationConfig()))
    else:
        assert d2_generation["settings"] == {"seed": 2026, "pair_count": 240, "positions": [32, 80], "matching_flanks": 6, "lesson": "fixed-position motif order, not universal CNN superiority"}
    d2_train = pd.read_csv(d2_dataset_dir / "train.csv")
    d2_validation = pd.read_csv(d2_dataset_dir / "validation.csv")
    assert d2_train.sample_id.tolist() == d2_record["training_ids"]
    assert d2_validation.sample_id.tolist() == d2_record["validation_ids"]
    assert d2_validation.sample_id.is_unique and len(d2_validation) == d2_size
    assert d2_validation.split.eq("validation").all()
    d2_data[d2_variant] = d2_validation
    d2_variant_metrics = d2_metrics.loc[d2_metrics.variant.eq(d2_variant)]
    assert len(d2_variant_metrics) == 3 and set(d2_variant_metrics.method) == set(d2_methods)
    assert d2_variant_metrics.evaluation_split.eq("validation").all()
    for d2_method in d2_methods:
        d2_saved = d2_predictions.loc[d2_predictions.variant.eq(d2_variant) & d2_predictions.method.eq(d2_method)]
        assert len(d2_saved) == d2_size and d2_saved.sample_id.is_unique
        d2_aligned = d2_validation[["sample_id", "group_id", "label"]].merge(d2_saved, on="sample_id", validate="one_to_one", suffixes=("", "_saved"))
        assert len(d2_aligned) == d2_size
        assert d2_aligned.label.eq(d2_aligned.label_saved).all()
        assert d2_aligned.group_id.eq(d2_aligned.group_id_saved).all()
        d2_recomputed = validation_metrics(d2_aligned.label, d2_aligned.probability)
        d2_metric = d2_variant_metrics.loc[d2_variant_metrics.method.eq(d2_method)].iloc[0]
        assert d2_metric.sample_count == d2_size and d2_metric.decision_threshold == 0.5
        for d2_key in ["roc_auc", "average_precision"]:
            np.testing.assert_allclose(d2_recomputed[d2_key], d2_metric[d2_key], rtol=0, atol=1e-12)
    d2_history = pd.read_csv(d2_root / "models" / d2_variant / "cnn_history.csv")
    assert d2_history.epoch.tolist() == list(range(1, 25))
    assert np.isfinite(d2_history.to_numpy()).all()
    d2_selected[d2_variant] = int(d2_history.loc[d2_history.validation_roc_auc.idxmax(), "epoch"])
    d2_histories[d2_variant] = d2_history
    d2_checkpoint = torch.load(d2_root / "models" / d2_variant / "cnn.pt", map_location="cpu", weights_only=True)
    assert d2_checkpoint["sequence_length"] == 160
    d2_model = SequenceCNN(160)
    d2_model.load_state_dict(d2_checkpoint["state_dict"])
    d2_models[d2_variant] = d2_model.eval()
    d2_saved_cnn = d2_predictions.loc[d2_predictions.variant.eq(d2_variant) & d2_predictions.method.eq("cnn")].set_index("sample_id").loc[d2_validation.sample_id, "probability"]
    np.testing.assert_allclose(predict_cnn(d2_model, d2_validation.sequence), d2_saved_cnn, rtol=0, atol=1e-6)
    d2_saved_auc = d2_variant_metrics.loc[d2_variant_metrics.method.eq("cnn"), "roc_auc"].iloc[0]
    np.testing.assert_allclose(d2_history.validation_roc_auc.max(), d2_saved_auc, rtol=0, atol=1e-12)
print("Source and generation identities checked; matching validation rows and metrics verified.")
print("Primary selected epoch:", d2_selected["simple"], "of 24 (earliest highest validation ROC-AUC)")
print("Primary metrics:")
print(d2_metrics.loc[d2_metrics.variant.eq("simple"), ["method", "roc_auc", "average_precision"]].to_string(index=False))
print("CPU aggregate primary-workflow seconds (all three methods):", round(d2_run["runs"]["simple"]["seconds"], 2))


In [ ]:
d2_history = d2_histories["simple"]
d2_figure, d2_axes = plt.subplots(1, 2, figsize=(11, 3.5))
for d2_axis, d2_key, d2_title, d2_color in zip(d2_axes, ["training_loss", "validation_roc_auc"], ["Training loss", "Validation ROC-AUC"], ["#0072B2", "#D55E00"]):
    d2_axis.plot(d2_history.epoch, d2_history[d2_key], color=d2_color, marker="o", markersize=3)
    d2_axis.axvline(d2_selected["simple"], color="#697782", linestyle="--", label="Selected epoch")
    d2_axis.set(xlabel="Epoch", ylabel=d2_title)
d2_axes[1].legend()
d2_figure.tight_layout()
plt.show()
plt.close(d2_figure)


### Inspect a prediction, then one response channel

Start at row 0 and channel 0. Change `d2_row_index` or `d2_channel` and rerun
**only the next cell**; fitted weights stay fixed. The stems show second-convolution
post-ReLU responses at sequence positions, not attribution or biological proof.
An error or a success on one record does not estimate general performance.


In [ ]:
d2_row_index = 0
d2_channel = 0
assert 0 <= d2_row_index < len(d2_data["simple"]) and 0 <= d2_channel < 16
d2_row = d2_data["simple"].iloc[d2_row_index]
with torch.inference_mode():
    d2_inputs = one_hot([d2_row.sequence])
    d2_probability = float(torch.sigmoid(d2_models["simple"](d2_inputs))[0])
    d2_response = d2_models["simple"].filters(d2_inputs)[0].numpy()
d2_reference = d2_predictions.loc[d2_predictions.variant.eq("simple") & d2_predictions.method.eq("cnn") & d2_predictions.sample_id.eq(d2_row.sample_id), "probability"].item()
np.testing.assert_allclose(d2_probability, d2_reference, rtol=0, atol=1e-6)
print("Validation ID:", d2_row.sample_id, "sequence:", d2_row.sequence)
print(f"P(B)={d2_probability:.6f}; observed {'B' if d2_row.label else 'A'}; fixed cutoff 0.5")
plt.figure(figsize=(11, 3))
plt.vlines(np.arange(160), 0, d2_response[d2_channel], color="#0072B2")
plt.xlabel("Position (zero-based)")
plt.ylabel(f"Channel {d2_channel} response")
plt.tight_layout()
plt.show()
plt.close("all")


## Day 2 · Section 6: same counts, different arrangement

The separate engineered control uses fixed TATAAATG / CGGACGTC blocks, matching
six-base flanks and shared backgrounds. Partners swap slots and labels but remain
in the same split. Complete overlapping four-mer equality forces equal scores for
any fixed bag-only predictor; balanced opposite-label pairs imply ROC-AUC 0.5.
Position-aware input retains a distinction, but actual learning is an observed result.

The stage 03 run already produced these models; **do not train again**.
This is a five-minute inspection, not a new architecture search.


In [ ]:
d2_group_index = 0  # Optional: change the group, rerun this cell only.
d2_control = pd.read_csv(d2_root / "models" / "interaction" / "controlled_order_changes.csv")
assert d2_control.sample_id.is_unique and len(d2_control) == 96
d2_pairs = d2_data["interaction"].merge(d2_control, on=["sample_id", "group_id", "label"], validate="one_to_one")
assert len(d2_pairs) == 96
d2_pair_groups = list(d2_pairs.groupby("group_id", sort=False))
assert len(d2_pair_groups) == 48
for d2_group_id, d2_pair in d2_pair_groups:
    assert len(d2_pair) == 2 and set(d2_pair.label) == {0, 1}
    d2_bags = [Counter(seq[i:i+4] for i in range(len(seq)-3)) for seq in d2_pair.sequence]
    assert d2_bags[0] == d2_bags[1] and sum(d2_bags[0].values()) == 157
    for d2_method in d2_methods:
        d2_reference = d2_predictions.loc[d2_predictions.variant.eq("interaction") & d2_predictions.method.eq(d2_method)].set_index("sample_id").loc[d2_pair.sample_id, "probability"]
        np.testing.assert_allclose(d2_pair[f"{d2_method}_probability"], d2_reference, rtol=0, atol=1e-6 if d2_method == "cnn" else 1e-12)
        if d2_method != "cnn":
            np.testing.assert_allclose(d2_pair[f"{d2_method}_probability"].iloc[0], d2_pair[f"{d2_method}_probability"].iloc[1], rtol=0, atol=1e-12)
for d2_method in d2_methods[:2]:
    np.testing.assert_allclose(validation_metrics(d2_pairs.label, d2_pairs[f"{d2_method}_probability"])["roc_auc"], 0.5, rtol=0, atol=1e-12)
assert 0 <= d2_group_index < len(d2_pair_groups)
d2_group_id, d2_pair = d2_pair_groups[d2_group_index]
print("Group:", d2_group_id, "— opposite labels; all 157 four-mer windows give identical complete Counters.")
print(d2_pair[["sample_id", "label", "logistic_regression_probability", "random_forest_probability", "cnn_probability"]].to_string(index=False))
for d2_record in d2_pair.itertuples():
    print("B" if d2_record.label else "A", d2_record.sequence)
print("Observed control metrics:")
print(d2_metrics.loc[d2_metrics.variant.eq("interaction"), ["method", "roc_auc", "average_precision"]].to_string(index=False))
print("Fixed-slot control only: no claim about arbitrary position, strand or biological grammar.")


# STOP — end of Day 2

Discuss what information your own task should retain. Keep this runtime and its
complete `course_output/` artifacts for Day 3; saving the notebook alone does not
save runtime files. The figures above are validation development evidence.
No test results have been calculated or used to guide a choice.

**Do not continue to stage 04 embeddings or stage 05 freeze today.**
Those belong to Day 3. Run all would cross this teaching boundary.
The next lesson asks whether we can reuse features learned before our task.


## Stage 04: Day 3: frozen DNA embeddings

Run the definition cell, then the workflow cell when the instructor reaches this stage. Definitions do not train models. Generated outputs are temporary runtime files.


In [ ]:
"""Frozen DNA embeddings with aligned caches and validation-only classification."""

from pathlib import Path
from time import perf_counter
from typing import Any
import hashlib
import json

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForMaskedLM, AutoTokenizer




def load_frozen_encoder() -> tuple[Any, Any]:
    torch.set_num_threads(2)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False, use_fast=False)
    model = AutoModelForMaskedLM.from_pretrained(MODEL_ID, revision=REVISION, code_revision=REVISION, trust_remote_code=True, use_safetensors=True)
    model.to(device="cpu", dtype=torch.float32)
    model.eval()
    model.requires_grad_(False)
    return tokenizer, model


def extract_embeddings(sequences: list[str], tokenizer: Any, model: Any, batch_size: int = 32) -> np.ndarray:
    if batch_size < 1 or not sequences or any(len(sequence) != 160 or set(sequence) - set("ACGT") for sequence in sequences):
        raise ValueError("Expected nonempty 160-bp ACGT sequences and a positive batch size")
    excluded = set(tokenizer.all_special_ids)
    batches = []
    with torch.inference_mode():
        for start in range(0, len(sequences), batch_size):
            encoded = tokenizer(sequences[start:start + batch_size], return_tensors="pt", padding=True, truncation=False)
            outputs = model(**encoded, output_hidden_states=True)
            mask = encoded["attention_mask"].bool()
            for token_id in excluded:
                mask &= encoded["input_ids"] != token_id
            if not mask.any(dim=1).all():
                raise ValueError("A sequence contains no poolable DNA tokens")
            hidden = outputs.hidden_states[-1]
            pooled = (hidden * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)
            batches.append(pooled.cpu().numpy())
    return np.concatenate(batches)


def run_foundation(output_dir: str | Path = "course_output", live_example: bool = True) -> pd.DataFrame:
    root = Path(output_dir)
    if any((root / marker).exists() for marker in ("frozen_manifest.json", "test_results.json")):
        raise RuntimeError("This experiment is frozen or revealed. Use a new development directory instead of tuning it.")
    results, records = [], {}
    tokenizer, model = None, None
    for variant in ("simple", "interaction"):
        samples = read_dataset(root / "datasets" / variant)
        cache_path = root / "embeddings" / f"{variant}.npz"
        started = perf_counter()
        if cache_path.exists():
            load_embedding_cache(cache_path, samples)
        extracted = not cache_path.exists()
        live_seconds = None
        live = None
        if live_example or extracted:
            if model is None:
                load_started = perf_counter()
                tokenizer, model = load_frozen_encoder()
                print(f"Encoder load: {perf_counter() - load_started:.2f}s, CPU")
            live_started = perf_counter()
            live_samples = samples.loc[samples["split"] == "train"].head(2)
            live = extract_embeddings(live_samples["sequence"].tolist(), tokenizer, model)
            live_seconds = perf_counter() - live_started
            print(f"{variant}: live two-sequence extraction {live.shape}, {live_seconds:.3f}s")
            write_json(root / f"{variant}_live_example.json", {"model_id": MODEL_ID, "revision": REVISION, "pooling": POOLING, "sample_ids": live_samples["sample_id"].tolist(), "shape": list(live.shape), "seconds": live_seconds, "device": "cpu", "cache_used_for_this_example": False})
        if extracted:
            embeddings = extract_embeddings(samples["sequence"].tolist(), tokenizer, model)
            if live is not None and not np.allclose(live, embeddings[:2], atol=1e-4, rtol=1e-4):
                raise ValueError("Live and batched extraction disagree")
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(cache_path, sample_ids=samples["sample_id"].to_numpy(dtype=str), sequence_sha256=sequence_hashes(samples), embeddings=embeddings, model_id=np.array(MODEL_ID), revision=np.array(REVISION), pooling=np.array(POOLING), dataset_generation_sha256=np.array(samples.attrs["dataset_generation_sha256"]))
        features = load_embedding_cache(cache_path, samples)
        extraction_seconds = perf_counter() - started
        train_mask = samples["split"] == "train"
        validation_mask = samples["split"] == "validation"
        classifier = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=2026))
        fit_started = perf_counter()
        classifier.fit(features[train_mask], samples.loc[train_mask, "label"])
        fit_seconds = perf_counter() - fit_started
        probabilities = classifier.predict_proba(features[validation_mask])[:, 1]
        metrics = validation_metrics(samples.loc[validation_mask, "label"], probabilities)
        results.append({"variant": variant, "method": "foundation", **metrics})
        destination = root / "models" / variant
        destination.mkdir(parents=True, exist_ok=True)
        joblib.dump(classifier, destination / "foundation.joblib")
        predictions = samples.loc[validation_mask, ["sample_id", "group_id", "label"]].copy()
        predictions["probability"] = probabilities
        predictions.to_csv(destination / "foundation_validation_predictions.csv", index=False)
        records[variant] = {"dataset_generation_sha256": file_hash(root / "datasets" / variant / "generation.json"), "cache": str(cache_path.relative_to(root)), "cache_created_this_run": extracted, "cache_load_or_extraction_seconds": extraction_seconds, "live_two_sequence_seconds": live_seconds, "classifier_fit_seconds": fit_seconds, "training_ids": samples.loc[train_mask, "sample_id"].tolist(), "validation_ids": samples.loc[validation_mask, "sample_id"].tolist(), "test_evaluated": False}
    result = pd.DataFrame(results)
    result.to_csv(root / "embedding_validation.csv", index=False)
    write_json(root / "embedding_run.json", {"model_id": MODEL_ID, "revision": REVISION, "pooling": POOLING, "license": "CC-BY-NC-SA-4.0", "device": "cpu", "torch_version": torch.__version__, "batch_size": 32, "runs": records, "limitation": "Frozen pretraining does not guarantee transfer to engineered motif order or real promoters. Pretraining overlap has not been audited."})
    print(result[["variant", "roc_auc", "average_precision"]].to_string(index=False))
    print("Validation only. Test labels were not scored. Full-data cache use and small live examples are recorded separately.")
    return result


In [ ]:
run_foundation()


## Stage 05: Day 3: freeze the comparison

Run the definition cell, then the workflow cell when the instructor reaches this stage. Definitions do not train models. Generated outputs are temporary runtime files.


In [ ]:
"""Freeze development artifacts, then explicitly reveal protected test results once."""

from pathlib import Path
from typing import Any, Callable
from datetime import datetime, timezone
import argparse
import ast
import hashlib
import inspect
import json
from importlib.metadata import version
from textwrap import dedent

import joblib
import numpy as np
import pandas as pd
import torch


REVEAL_CONFIRMATION = "REVEAL THE FROZEN TEST RESULTS"


def artifact_paths(root: Path) -> list[Path]:
    paths = [root / name for name in ("representation_run.json", "representation_validation.csv", "representation_validation_predictions.csv", "embedding_run.json", "embedding_validation.csv")]
    for variant in ("simple", "interaction"):
        paths.extend(root / "datasets" / variant / name for name in ("train.csv", "validation.csv", "protected_test.csv", "split_manifest.csv", "simulation_truth.csv", "generation.json"))
        paths.extend(root / "models" / variant / name for name in ("logistic_regression.joblib", "random_forest.joblib", "cnn.pt", "cnn_history.csv", "foundation.joblib"))
        paths.append(root / "embeddings" / f"{variant}.npz")
        paths.append(root / "models" / variant / "foundation_validation_predictions.csv")
    paths.append(root / "models" / "interaction" / "controlled_order_changes.csv")
    return paths


def inference_identity() -> dict[str, Any]:
    functions: dict[str, Callable[..., Any]] = {"cnn_init": SequenceCNN.__init__, "cnn_forward": SequenceCNN.forward, "one_hot": one_hot, "predict_cnn": predict_cnn, "sequence_hashes": sequence_hashes, "load_embedding_cache": load_embedding_cache, "validate_generation": validate_generation, "generation_contract": generation_contract, "settings_fingerprint": settings_fingerprint, "simulation_settings": simulation_settings, "read_dataset": read_dataset, "classification_metrics": classification_metrics, "reveal_test": reveal_test}
    fingerprints = {}
    for name, function in functions.items():
        try:
            syntax = ast.dump(ast.parse(dedent(inspect.getsource(function))), include_attributes=False)
        except (OSError, TypeError) as error:
            raise RuntimeError("Cannot capture inference source. Run a script or a notebook cell that preserves source history before freezing.") from error
        fingerprints[name] = hashlib.sha256(syntax.encode("utf-8")).hexdigest()
    return {"source_sha256": fingerprints, "packages": {name: version(name) for name in ("numpy", "pandas", "scikit-learn", "torch")}}


def write_exclusive(path: Path, value: Any) -> None:
    with path.open("x", encoding="utf-8") as output:
        output.write(json.dumps(value, indent=2, allow_nan=False) + "\n")


def freeze_workflows(output_dir: str | Path = "course_output") -> dict[str, Any]:
    root = Path(output_dir)
    if (root / "test_results.json").exists():
        raise RuntimeError("Test results already exist; this is not a new independent evaluation")
    if (root / "frozen_manifest.json").exists():
        raise FileExistsError("A freeze already exists. Keep it unchanged or start in a new development directory.")
    representation = json.loads((root / "representation_run.json").read_text(encoding="utf-8"))
    embedding = json.loads((root / "embedding_run.json").read_text(encoding="utf-8"))
    for variant in ("simple", "interaction"):
        samples = read_dataset(root / "datasets" / variant)
        load_embedding_cache(root / "embeddings" / f"{variant}.npz", samples)
        for record in (representation["runs"][variant], embedding["runs"][variant]):
            if record.get("dataset_generation_sha256") != samples.attrs["dataset_generation_sha256"]:
                raise ValueError("Model generation identity mismatch; regenerate development artifacts")
            if record["test_evaluated"]:
                raise ValueError("Development metadata reports previous test evaluation")
            for split, key in (("train", "training_ids"), ("validation", "validation_ids")):
                if sorted(record[key]) != sorted(samples.loc[samples["split"] == split, "sample_id"]):
                    raise ValueError("Model development identities differ from the dataset")
    fingerprints = {path.relative_to(root).as_posix(): file_hash(path) for path in artifact_paths(root)}
    manifest = {"generation_contract": generation_contract(), "created_utc": datetime.now(timezone.utc).isoformat(), "primary_metric": "roc_auc", "decision_threshold": 0.5, "methods": ["logistic_regression", "random_forest", "cnn", "foundation"], "artifacts": fingerprints, "inference_identity": inference_identity(), "policy": "Freeze all four workflows before test reveal. No post-reveal tuning is independent test evidence. Hashes enforce an accident guard, not adversarial access security."}
    write_exclusive(root / "frozen_manifest.json", manifest)
    print("Frozen. Test results have NOT been calculated. Discuss expected outcomes before revealing.")
    return manifest


def reveal_test(output_dir: str | Path = "course_output", confirmation: str = "") -> pd.DataFrame:
    if confirmation != REVEAL_CONFIRMATION:
        raise ValueError("Explicit confirmation is required to reveal protected test results")
    root = Path(output_dir)
    if (root / "test_results.json").exists():
        raise FileExistsError("Results have already been revealed. Read the saved results; do not retune and repeat.")
    manifest = json.loads((root / "frozen_manifest.json").read_text(encoding="utf-8"))
    if settings_fingerprint(manifest.get("generation_contract", {})) != settings_fingerprint(generation_contract()):
        raise ValueError("Incompatible frozen generator contract; keep historical evidence and regenerate development separately")
    if manifest["inference_identity"] != inference_identity():
        raise ValueError("Frozen inference implementation or package versions changed")
    expected_paths = {path.relative_to(root).as_posix() for path in artifact_paths(root)}
    if set(manifest["artifacts"]) != expected_paths:
        raise ValueError("Frozen artifact inventory is incomplete or unexpected")
    for relative, expected_hash in manifest["artifacts"].items():
        if file_hash(root / relative) != expected_hash:
            raise ValueError(f"Frozen artifact changed: {relative}")
    results, predictions = [], []
    for variant in ("simple", "interaction"):
        samples = read_dataset(root / "datasets" / variant, include_test_labels=True)
        test = samples.loc[samples["split"] == "test"]
        destination = root / "models" / variant
        probabilities_by_method = {}
        for method in ("logistic_regression", "random_forest"):
            classifier = joblib.load(destination / f"{method}.joblib")
            probabilities_by_method[method] = classifier.predict_proba(test["sequence"])[:, 1]
        checkpoint = torch.load(destination / "cnn.pt", map_location="cpu", weights_only=True)
        cnn = SequenceCNN(checkpoint["sequence_length"])
        cnn.load_state_dict(checkpoint["state_dict"])
        probabilities_by_method["cnn"] = predict_cnn(cnn, test["sequence"])
        features = load_embedding_cache(root / "embeddings" / f"{variant}.npz", samples)
        foundation = joblib.load(destination / "foundation.joblib")
        probabilities_by_method["foundation"] = foundation.predict_proba(features[samples["split"] == "test"])[:, 1]
        for method, probabilities in probabilities_by_method.items():
            metrics = classification_metrics(test["label"], probabilities, "test")
            results.append({"variant": variant, "method": method, **metrics})
            frame = test[["sample_id", "group_id", "label"]].copy()
            frame["probability"] = probabilities
            frame["variant"], frame["method"] = variant, method
            predictions.append(frame)
    write_exclusive(root / "test_results.json", {"revealed_utc": datetime.now(timezone.utc).isoformat(), "frozen_manifest_sha256": file_hash(root / "frozen_manifest.json"), "results": results, "interpretation": "One planned test comparison of validation-developed workflows, not a biological benchmark. Do not select changes using these test scores."})
    pd.concat(predictions, ignore_index=True).to_csv(root / "test_predictions.csv", index=False)
    table = pd.DataFrame(results)
    table.to_csv(root / "test_comparison.csv", index=False)
    print(table[["variant", "method", "roc_auc", "average_precision"]].to_string(index=False))
    return table


In [ ]:
freeze_workflows()
